# Unified Crawling Data All Polutan Sentinel-5P (openEO)

Notebook terpadu ini mengambil data konsentrasi **CH4, CO, NO2, dan SO2** dari satelit Sentinel-5P melalui platform **openEO (Copernicus Data Space Ecosystem)** untuk area observasi polygon baru (BBOX), lalu menyimpannya dalam format NetCDF (`.nc`) serta mengekspornya ke CSV individual dan CSV terpadu (`polutan_gresik.csv`).

In [1]:
import openeo
import netCDF4
import pandas as pd
import os

## 1. Autentikasi Koneksi openEO

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


## 2. Area of Interest (AOI) & Spatial Extent

Definisi polygon GeoJSON lokasi pengamatan baru dan Bounding Box (BBOX).

In [3]:
# GeoJSON Polygon Baru
aoi = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [112.5058378, -7.0230496],
            [112.5963518, -7.0230496],
            [112.5963518, -7.0683846],
            [112.5067768, -7.0683846],
            [112.5058378, -7.0230496]
          ]
        ]
      }
    }
  ]
}

# Bounding Box Spasial
spatial_extent = {
    "west": 112.5058378,
    "south": -7.0683846,
    "east": 112.5963518,
    "north": -7.0230496
}

# Periode Pengamatan Harian (1 Tahun)
start_date = "2025-08-24"
end_date = "2026-08-23"

## 3. Crawling & Processing Terpadu (CH4, CO, NO2, SO2)

In [4]:
# Daftar Polutan yang akan di-crawl
pollutants = ["CH4", "CO", "NO2", "SO2"]

# Pastikan directory data output ada
os.makedirs("../data/nc", exist_ok=True)
os.makedirs("../data/csv", exist_ok=True)

# Buat Dataframe Master Rentang Tanggal Lengkap
full_dates = pd.date_range(start=start_date, end=end_date, freq="D")
master_df = pd.DataFrame({"date": full_dates.strftime("%Y-%m-%d")})

# Iterasi pengambilan data untuk setiap polutan
for p in pollutants:
    print(f"\n=========================================")
    print(f"[*] Memulai Proses Crawling Polutan: {p}")
    print(f"=========================================")
    
    # Load collection Sentinel-5P L2
    s5 = connection.load_collection(
        "SENTINEL_5P_L2",
        temporal_extent=[start_date, end_date],
        spatial_extent=spatial_extent,
        bands=[p]
    )
    
    # Agregasi Spasial (mean polygon) & Temporal (mean harian)
    s5 = s5.aggregate_temporal_period(reducer="mean", period="day")
    s5 = s5.aggregate_spatial(reducer="mean", geometries=aoi)
    
    nc_path = f"../data/nc/polutan_{p}_gresik.nc"
    csv_path = f"../data/csv/{p}_gresik_timeseries.csv"
    
    # Eksekusi Batch Job di Server openEO
    print(f"Mengirim Batch Job openEO untuk {p}...")
    job = s5.execute_batch(title=f"{p} Gresik Crawling", outputfile=nc_path)
    print(f"[✓] File NetCDF tersimpan di: {nc_path}")
    
    # Buka NetCDF dan Ekstrak ke Dataframe
    ds = netCDF4.Dataset(nc_path)
    vals = ds.variables[p][0, :]
    time_vals = ds.variables["t"][:]
    dates = netCDF4.num2date(time_vals, units=ds.variables["t"].units)
    
    p_data = {d.strftime("%Y-%m-%d"): float(val) for d, val in zip(dates, vals)}
    
    # Simpan CSV Individual
    df_single = pd.DataFrame({"date": master_df["date"]})
    df_single[p] = df_single["date"].map(p_data)
    df_single.to_csv(csv_path, index=False)
    print(f"[✓] File CSV Individual tersimpan di: {csv_path}")
    
    # Gabungkan ke Master Dataframe
    master_df[p] = master_df["date"].map(p_data)
    ds.close()

# Simpan CSV Master Gabungan Seluruh Polutan
master_csv_path = "../data/csv/polutan_gresik.csv"
master_df.to_csv(master_csv_path, index=False)
print(f"\n[SUCCESS] Master CSV Terpadu (CH4, CO, NO2, SO2) Berhasil Disimpan: {master_csv_path}")


[*] Memulai Proses Crawling Polutan: CH4


Mengirim Batch Job openEO untuk CH4...


0:00:00 Job 'j-260911033913451eaf0ea0a7af47c80e': send 'start'


0:00:04 Job 'j-260911033913451eaf0ea0a7af47c80e': queued (progress 0%)


0:00:09 Job 'j-260911033913451eaf0ea0a7af47c80e': queued (progress 0%)


0:00:16 Job 'j-260911033913451eaf0ea0a7af47c80e': queued (progress 0%)


0:00:24 Job 'j-260911033913451eaf0ea0a7af47c80e': queued (progress 0%)


## 4. Pratinjau Data Terpadu

In [5]:
# Pratinjau Master Dataframe
print("Shape Master Data:", master_df.shape)
display(master_df.head(10))
display(master_df.info())

Shape Master Data: (366, 5)


,date,CH4,CO,NO2,SO2
0,2025-08-24,NaN,0.031251,0.000081,0.000271
1,2025-08-25,NaN,0.034601,0.000134,0.000586
2,2025-08-26,NaN,NaN,0.000069,0.001380
3,2025-08-27,NaN,0.028711,0.000056,0.000094
4,2025-08-28,NaN,0.028990,0.000080,-0.000080
5,2025-08-29,1871.754028,0.022500,0.000085,0.000466
6,2025-08-30,NaN,0.026755,0.000048,0.000148
7,2025-08-31,NaN,0.029416,0.000034,-0.000571
8,2025-09-01,NaN,0.024829,0.000049,-0.000355
9,2025-09-02,NaN,0.023791,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    366 non-null    object 
 1   CH4     20 non-null     float64
 2   CO      192 non-null    float64
 3   NO2     200 non-null    float64
 4   SO2     235 non-null    float64
dtypes: float64(4), object(1)
memory usage: 14.4+ KB


None